# Taller Sumativo N°2: Análisis de Reglas de Asociación en Customer Churn

**Universidad:** Universidad San Sebastián  
**Asignatura:** Introducción a la Ciencia de Datos  
**Unidad:** 2 - Ciencia de Datos  
**Carácter:** Grupal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías importadas correctamente')

## 1️⃣ CARGA Y EXPLORACIÓN DE DATOS

In [ ]:
# Cargar dataset
try:
    df = pd.read_csv('customer_churn.csv')
    print(f'✓ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
except FileNotFoundError:
    print('⚠️ Descargar desde: https://www.kaggle.com/datasets/yousefhossam2222/customer-churn-dataset-csv')
    df = None

In [ ]:
if df is not None:
    print('\nPRIMERAS FILAS:')
    print(df.head())
    print('\nTIPOS DE DATOS:')
    print(df.dtypes)
    print('\nVALORES NULOS:')
    print(df.isnull().sum())
    print('\nESTADÍSTICAS:')
    print(df.describe())

## 2️⃣ LIMPIEZA Y TRANSFORMACIÓN

In [ ]:
if df is not None:
    df_transformed = df.copy()
    
    # Discretizar Age
    df_transformed['Age_Group'] = pd.cut(
        df['Age'],
        bins=[0, 25, 40, 55, 100],
        labels=['Joven (18-25)', 'Adulto (26-40)', 'Maduro (41-55)', 'Mayor (56+)']
    )
    print('✓ Age discretizado')
    
    # Discretizar Tenure
    df_transformed['Tenure_Group'] = pd.cut(
        df['Tenure'],
        bins=[-1, 6, 24, 60, 100],
        labels=['Nuevo (0-6m)', 'Corto (7-24m)', 'Medio (25-60m)', 'Largo (60+ m)']
    )
    print('✓ Tenure discretizado')
    
    # Discretizar Total Spend
    df_transformed['Spend_Group'] = pd.cut(
        df['Total Spend'],
        bins=[-1, 500, 1500, 3000, 10000],
        labels=['Bajo (<$500)', 'Medio ($500-$1500)', 'Alto ($1500-$3000)', 'Muy Alto (>$3000)']
    )
    print('✓ Total Spend discretizado')

In [ ]:
if df is not None:
    # Seleccionar variables para análisis
    variables = ['Age_Group', 'Gender', 'Tenure_Group', 'Usage Frequency', 'Spend_Group', 'Churn']
    df_analisis = df_transformed[variables].copy()
    
    # One-Hot Encoding
    df_encoded = pd.get_dummies(df_analisis, drop_first=False)
    
    print(f'✓ One-Hot Encoding completado')
    print(f'  Dimensiones: {df_encoded.shape[0]} × {df_encoded.shape[1]}')
    print(f'\nPrimeras 5 filas:')
    print(df_encoded.head())

## 3️⃣ ALGORITMO APRIORI

In [ ]:
if df is not None:
    print('Ejecutando Apriori...')
    
    # Parámetros
    min_support = 0.02
    min_confidence = 0.30
    
    # Itemsets frecuentes
    frequent_itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
    print(f'✓ Itemsets frecuentes encontrados: {len(frequent_itemsets)}')
    
    # Reglas
    rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=min_confidence)
    print(f'✓ Reglas generadas: {len(rules)}')
    
    # Ordenar por lift
    rules_sorted = rules.sort_values('lift', ascending=False)
    print(f'\nTop 10 reglas por Lift:')
    print(rules_sorted[['support', 'confidence', 'lift']].head(10))

## 4️⃣ INTERPRETACIÓN DE REGLAS

In [ ]:
if df is not None and len(rules) > 0:
    print('ANÁLISIS DE LAS 3 MEJORES REGLAS')
    print('='*80)
    
    top_3 = rules_sorted.head(3)
    
    for idx, (_, rule) in enumerate(top_3.iterrows(), 1):
        ant = list(rule['antecedents'])
        cons = list(rule['consequents'])
        
        print(f'\nREGLA #{idx}')
        print(f'  SI: {ant}')
        print(f'  ENTONCES: {cons}')
        print(f'  Soporte: {rule["support"]:.4f} ({rule["support"]*100:.2f}%)')
        print(f'  Confianza: {rule["confidence"]:.4f} ({rule["confidence"]*100:.2f}%)')
        print(f'  Lift: {rule["lift"]:.4f}')

## 5️⃣ VISUALIZACIONES

In [ ]:
if df is not None and len(rules) > 0:
    # Gráfico 1: Scatter Soporte vs Confianza
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(rules['support'], rules['confidence'], 
                       c=rules['lift'], s=100, alpha=0.6, cmap='RdYlGn')
    ax.set_xlabel('Soporte')
    ax.set_ylabel('Confianza')
    ax.set_title('Soporte vs Confianza (color=Lift)')
    plt.colorbar(scatter, ax=ax)
    plt.tight_layout()
    plt.show()
    
    print('✓ Gráfico 1 completado')

In [ ]:
if df is not None and len(rules) > 0:
    # Gráfico 2: Distribución Lift
    plt.figure(figsize=(10, 5))
    plt.hist(rules['lift'], bins=30, color='#9b59b6', alpha=0.7)
    plt.axvline(x=1, color='red', linestyle='--', label='Independencia (Lift=1)')
    plt.xlabel('Lift')
    plt.ylabel('Frecuencia')
    plt.title('Distribución del Lift')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    print('✓ Gráfico 2 completado')

## CONCLUSIONES Y RECOMENDACIONES

### Hallazgos Principales
Este análisis ha identificado patrones significativos de churn mediante reglas de asociación.

### Recomendaciones
1. Implementar programas de retención para los segmentos de alto riesgo
2. Monitorear continuamente estos patrones
3. Evaluar ROI de las iniciativas implementadas
4. Actualizar el análisis con nuevos datos mensualmente